# Análisis de Similitud de Malware — Índice de Jaccard Modificado

- Juan Pablo Solis
- Andre Yatmian Jo Mai

Se utiliza el **Índice de Jaccard Modificado** para medir similitud entre muestras de malware usando dos características:

1. **Strings** — sub-tokens extraídos descomponiendo los nombres de API calls (CamelCase splitting)
2. **Llamadas a funciones (API calls)** — conjunto de APIs únicas por muestra

Para cada característica se analizan **tres umbrales de similitud** (0.3, 0.5, 0.7) y se generan:
- Un grafo por familia/clase de malware
- Un grafo para todo el conjunto de muestras

## 1. Imports y Configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import re
import warnings
warnings.filterwarnings('ignore')

# ── Paleta de colores por familia ─────────────────────────────────────────────
FAMILY_COLORS = [
    '#1f77b4', '#d62728', '#ff7f0e', '#2ca02c',
    '#9467bd', '#8c564b', '#e377c2', '#17becf'
]

# ── Umbrales de similitud ─────────────────────────────────────────────────────
THRESHOLDS = [0.3, 0.5, 0.7]

print('Imports OK')

Imports OK


## 2. Carga del Dataset

Se usa el dataset **MalBehavD-V1** con columnas:
- `sha256` — hash de la muestra
- `labels` — 0 = Benigno, 1 = Malware
- Columnas `0..N` — secuencia de API calls

In [ ]:
DATA_PATH = 'malware_dataset.csv'

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print('Distribución de labels:')
print(df['labels'].value_counts())
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'MalBehavD-V1-dataset.csv'

## 3. Preprocesamiento — Construcción de Conjuntos

Se crean dos representaciones por muestra:

- **`api_set`**: conjunto de API calls únicas (sin orden). Ej: `{NtOpenKey, NtClose, RegQueryValueExA}`
- **`string_set`**: sub-tokens obtenidos descomponiendo los nombres de API calls por CamelCase. Ej: `{nt, open, key, close, reg, query, value, ex}`

El índice de Jaccard opera sobre conjuntos, no sobre secuencias ordenadas.

In [ ]:
# Columnas de API calls
META_COLS = ['sha256', 'labels']
api_cols  = [c for c in df.columns if c not in META_COLS]

# Mapeo de etiqueta numérica a nombre de familia legible
df['family'] = df['labels'].map({0: 'Benigno', 1: 'Malware'})

def build_api_set(row):
    """Conjunto de API calls únicas de una fila (ignora NaN y vacíos)."""
    return frozenset(
        str(v).strip()
        for v in row[api_cols]
        if pd.notna(v) and str(v).strip() != ''
    )

def build_string_set(api_set):
    """
    Característica 'strings':
    Descompone los nombres de API calls usando CamelCase y guiones bajos.
    Ej: 'NtOpenKey' → {'nt', 'open', 'key'}
        'RegQueryValueExA' → {'reg', 'query', 'value', 'ex', 'a'}
    """
    tokens = set()
    for api in api_set:
        # Insertar espacio antes de cada mayúscula que sigue a minúscula
        parts = re.sub(r'([a-z])([A-Z])', r'\1 \2', api).split()
        parts += api.split('_')
        tokens.update(p.lower() for p in parts if len(p) > 1)
    return frozenset(tokens)

print('Construyendo conjuntos...')
df['api_set']    = df.apply(build_api_set, axis=1)
df['string_set'] = df['api_set'].apply(build_string_set)

print(f'Muestras totales : {len(df)}')
print(f'Familias         : {df["family"].value_counts().to_dict()}')
print(f'\nEjemplo api_set    (muestra 0): {list(df["api_set"].iloc[0])[:5]}')
print(f'Ejemplo string_set (muestra 0): {list(df["string_set"].iloc[0])[:8]}')

## 4. Índice de Jaccard Modificado

La versión **modificada** penaliza menos las diferencias de tamaño entre conjuntos usando el parámetro $\alpha$:

$$J_{mod}(A, B) = \frac{|A \cap B|}{\min(|A|, |B|) + \alpha \cdot (|A \cup B| - |A \cap B|)}$$

- Con $\alpha = 1.0$ → Jaccard estándar
- Con $\alpha = 0.5$ → versión modificada (menos penalización por diferencias de tamaño)

El resultado está siempre en $[0, 1]$: 0 = sin similitud, 1 = idénticos.

In [ ]:
def jaccard_modificado(set_a: frozenset, set_b: frozenset, alpha: float = 0.5) -> float:
    """
    Índice de Jaccard Modificado entre dos conjuntos.

    Parámetros
    ----------
    set_a, set_b : frozenset
        Conjuntos de características a comparar.
    alpha : float (0 < alpha <= 1)
        Factor de penalización.
        alpha=1.0  → Jaccard estándar
        alpha=0.5  → versión modificada (menos penalización por tamaño)

    Retorna
    -------
    float en [0, 1]
    """
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0

    interseccion = len(set_a & set_b)
    union        = len(set_a | set_b)
    minimo       = min(len(set_a), len(set_b))

    denominador = minimo + alpha * (union - interseccion)
    return interseccion / denominador if denominador > 0 else 0.0


# ── Verificación ──────────────────────────────────────────────────────────────
A = frozenset({'NtOpenKey', 'NtClose', 'RegQueryValueExA'})
B = frozenset({'NtOpenKey', 'NtClose', 'LdrUnloadDll', 'GetSystemInfo', 'RegQueryValueExA'})

print('Prueba Jaccard Modificado:')
print(f'  A = {A}')
print(f'  B = {B}')
print(f'  J estándar   (α=1.0): {jaccard_modificado(A, B, alpha=1.0):.4f}')
print(f'  J modificado (α=0.5): {jaccard_modificado(A, B, alpha=0.5):.4f}')
print()
print('Nota: el modificado da un valor más alto porque penaliza menos')
print('la diferencia de tamaño entre A y B.')

## 5. Funciones de Construcción y Visualización de Grafos

- **Nodo** = muestra de malware
- **Arista** = similitud ≥ umbral entre dos muestras
- **Grosor de arista** = valor de similitud
- **Tamaño de nodo** = proporcional al grado (nodos más conectados = más grandes)

In [ ]:
def build_graph(sample_ids, sets, threshold, alpha=0.5, node_labels=None):
    """
    Construye un grafo de similitud.

    Parámetros
    ----------
    sample_ids  : lista de identificadores únicos para los nodos
    sets        : lista de frozensets (misma longitud que sample_ids)
    threshold   : umbral mínimo de similitud para agregar arista
    alpha       : parámetro del Jaccard modificado
    node_labels : dict {id: etiqueta} para visualización
    """
    G = nx.Graph()
    n = len(sample_ids)

    for sid in sample_ids:
        lbl = node_labels.get(sid, str(sid)) if node_labels else str(sid)
        G.add_node(sid, label=lbl)

    for i in range(n):
        for j in range(i + 1, n):
            sim = jaccard_modificado(sets[i], sets[j], alpha)
            if sim >= threshold:
                G.add_edge(sample_ids[i], sample_ids[j], weight=sim)
    return G


def plot_graph(G, title, node_color='#1f77b4', ax=None):
    """Visualiza un grafo de similitud en un Axes de matplotlib."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 5))

    if G.number_of_nodes() == 0:
        ax.text(0.5, 0.5, 'Sin muestras', ha='center', va='center',
                transform=ax.transAxes, fontsize=10, color='gray')
        ax.set_title(title, fontsize=9)
        ax.axis('off')
        return

    pos = nx.spring_layout(G, seed=42, k=1.2)

    degrees    = dict(G.degree())
    node_sizes = [120 + degrees[n] * 70 for n in G.nodes()]

    edges       = list(G.edges())
    edge_widths = [G[u][v]['weight'] * 3.5 for u, v in edges] if edges else []

    nx.draw_networkx_nodes(G, pos, node_size=node_sizes,
                           node_color=node_color, alpha=0.88, ax=ax)
    nx.draw_networkx_edges(G, pos, width=edge_widths,
                           edge_color='#555555', alpha=0.45, ax=ax)
    nx.draw_networkx_labels(G, pos,
                            labels={n: G.nodes[n]['label'][:7] for n in G.nodes()},
                            font_size=5.5, ax=ax)

    n_nodos  = G.number_of_nodes()
    n_aristas = G.number_of_edges()
    densidad = nx.density(G) if n_nodos > 1 else 0.0

    ax.set_title(
        f'{title}\n'
        f'Nodos={n_nodos}  Aristas={n_aristas}  Densidad={densidad:.3f}',
        fontsize=8.5
    )
    ax.axis('off')


print('Funciones de grafos definidas correctamente.')

---
# Característica 1: API Calls (Llamadas a Funciones)

Cada muestra se representa como el **conjunto de API calls únicas** que realiza.

## 6a. API Calls — Grafo por Familia (los 3 umbrales)

In [ ]:
families = sorted(df['family'].unique())
n_fam    = len(families)

# ── Una figura por umbral ──────────────────────────────────────────────────────
for threshold in THRESHOLDS:
    ncols = n_fam
    fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 6))
    if ncols == 1:
        axes = [axes]

    fig.suptitle(
        f'API Calls — Grafo por Familia  |  Umbral = {threshold}  (α = 0.5)',
        fontsize=13, fontweight='bold'
    )

    for k, family in enumerate(families):
        sub = df[df['family'] == family].reset_index()
        ids    = list(sub['index'])                   # índices originales como IDs
        sets   = list(sub['api_set'])
        labels = {sub.loc[i, 'index']: str(sub.loc[i, 'sha256'])[:8]
                  for i in range(len(sub))}

        G = build_graph(ids, sets, threshold, alpha=0.5, node_labels=labels)
        color = FAMILY_COLORS[k % len(FAMILY_COLORS)]
        plot_graph(G, title=f'Familia: {family}', node_color=color, ax=axes[k])

    plt.tight_layout()
    plt.savefig(f'api_calls_familia_umbral_{int(threshold*10)}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Umbral {threshold} generado.')

## 6b. API Calls — Grafo de Todo el Conjunto (los 3 umbrales)

In [ ]:
family_to_color = {f: FAMILY_COLORS[i % len(FAMILY_COLORS)]
                   for i, f in enumerate(families)}

for threshold in THRESHOLDS:
    fig, ax = plt.subplots(figsize=(14, 10))

    all_ids   = list(df.index)
    all_sets  = list(df['api_set'])
    all_lbl   = {i: str(df.loc[i, 'sha256'])[:8] for i in all_ids}

    G_all = build_graph(all_ids, all_sets, threshold, alpha=0.5, node_labels=all_lbl)

    # Color de nodo según familia
    node_colors = [family_to_color[df.loc[n, 'family']] for n in G_all.nodes()]

    pos = nx.spring_layout(G_all, seed=42, k=0.6)
    degrees    = dict(G_all.degree())
    node_sizes = [80 + degrees[n] * 35 for n in G_all.nodes()]
    edges       = list(G_all.edges())
    edge_widths = [G_all[u][v]['weight'] * 2 for u, v in edges] if edges else []

    nx.draw_networkx_nodes(G_all, pos, node_size=node_sizes,
                           node_color=node_colors, alpha=0.85, ax=ax)
    nx.draw_networkx_edges(G_all, pos, width=edge_widths,
                           edge_color='gray', alpha=0.3, ax=ax)

    # Leyenda de familias
    legend_patches = [
        mpatches.Patch(color=family_to_color[f], label=f)
        for f in families
    ]
    ax.legend(handles=legend_patches, loc='upper left', fontsize=10,
              title='Familia', title_fontsize=10)

    n_nodos   = G_all.number_of_nodes()
    n_aristas = G_all.number_of_edges()
    densidad  = nx.density(G_all) if n_nodos > 1 else 0
    ax.set_title(
        f'API Calls — Todo el Conjunto  |  Umbral = {threshold}  (α = 0.5)\n'
        f'Nodos={n_nodos}  Aristas={n_aristas}  Densidad={densidad:.4f}',
        fontsize=12, fontweight='bold'
    )
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'api_calls_total_umbral_{int(threshold*10)}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Umbral {threshold}: {n_nodos} nodos, {n_aristas} aristas, densidad={densidad:.4f}')

---
# Característica 2: Strings

Cada muestra se representa como el **conjunto de sub-tokens** obtenidos al descomponer los nombres de las API calls por CamelCase.

Ejemplo: `NtOpenKey` → `{nt, open, key}` · `RegQueryValueExA` → `{reg, query, value, ex}`

Esta representación captura similitudes a nivel de **componentes léxicos** en lugar de API calls exactas, lo que permite detectar similitudes entre muestras que usan funciones relacionadas (misma familia de prefijos `Nt*`, `Reg*`, etc.).

## 7a. Strings — Grafo por Familia (los 3 umbrales)

In [ ]:
for threshold in THRESHOLDS:
    ncols = n_fam
    fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 6))
    if ncols == 1:
        axes = [axes]

    fig.suptitle(
        f'Strings — Grafo por Familia  |  Umbral = {threshold}  (α = 0.5)',
        fontsize=13, fontweight='bold'
    )

    for k, family in enumerate(families):
        sub = df[df['family'] == family].reset_index()
        ids    = list(sub['index'])
        sets   = list(sub['string_set'])
        labels = {sub.loc[i, 'index']: str(sub.loc[i, 'sha256'])[:8]
                  for i in range(len(sub))}

        G = build_graph(ids, sets, threshold, alpha=0.5, node_labels=labels)
        color = FAMILY_COLORS[k % len(FAMILY_COLORS)]
        plot_graph(G, title=f'Familia: {family}', node_color=color, ax=axes[k])

    plt.tight_layout()
    plt.savefig(f'strings_familia_umbral_{int(threshold*10)}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Umbral {threshold} generado.')

## 7b. Strings — Grafo de Todo el Conjunto (los 3 umbrales)

In [ ]:
for threshold in THRESHOLDS:
    fig, ax = plt.subplots(figsize=(14, 10))

    all_ids  = list(df.index)
    all_sets = list(df['string_set'])
    all_lbl  = {i: str(df.loc[i, 'sha256'])[:8] for i in all_ids}

    G_all = build_graph(all_ids, all_sets, threshold, alpha=0.5, node_labels=all_lbl)

    node_colors = [family_to_color[df.loc[n, 'family']] for n in G_all.nodes()]

    pos = nx.spring_layout(G_all, seed=42, k=0.6)
    degrees    = dict(G_all.degree())
    node_sizes = [80 + degrees[n] * 35 for n in G_all.nodes()]
    edges       = list(G_all.edges())
    edge_widths = [G_all[u][v]['weight'] * 2 for u, v in edges] if edges else []

    nx.draw_networkx_nodes(G_all, pos, node_size=node_sizes,
                           node_color=node_colors, alpha=0.85, ax=ax)
    nx.draw_networkx_edges(G_all, pos, width=edge_widths,
                           edge_color='gray', alpha=0.3, ax=ax)

    legend_patches = [
        mpatches.Patch(color=family_to_color[f], label=f)
        for f in families
    ]
    ax.legend(handles=legend_patches, loc='upper left', fontsize=10,
              title='Familia', title_fontsize=10)

    n_nodos   = G_all.number_of_nodes()
    n_aristas = G_all.number_of_edges()
    densidad  = nx.density(G_all) if n_nodos > 1 else 0
    ax.set_title(
        f'Strings — Todo el Conjunto  |  Umbral = {threshold}  (α = 0.5)\n'
        f'Nodos={n_nodos}  Aristas={n_aristas}  Densidad={densidad:.4f}',
        fontsize=12, fontweight='bold'
    )
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'strings_total_umbral_{int(threshold*10)}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Umbral {threshold}: {n_nodos} nodos, {n_aristas} aristas, densidad={densidad:.4f}')

---
## 8. Resumen Comparativo

Tabla resumen de métricas del grafo (nodos, aristas, densidad) para cada combinación de característica × umbral.

In [ ]:
rows = []

for feat_name, feat_col in [('API Calls', 'api_set'), ('Strings', 'string_set')]:
    for threshold in THRESHOLDS:
        # ── Global ────────────────────────────────────────────────────────────
        all_ids  = list(df.index)
        all_sets = list(df[feat_col])
        G_total  = build_graph(all_ids, all_sets, threshold, alpha=0.5)
        n_n = G_total.number_of_nodes()
        n_e = G_total.number_of_edges()
        dens = nx.density(G_total) if n_n > 1 else 0
        rows.append({
            'Característica': feat_name,
            'Umbral': threshold,
            'Alcance': 'Global',
            'Familia': '—',
            'Nodos': n_n,
            'Aristas': n_e,
            'Densidad': round(dens, 4),
        })

        # ── Por familia ────────────────────────────────────────────────────────
        for family in families:
            sub   = df[df['family'] == family].reset_index()
            ids   = list(sub['index'])
            sets  = list(sub[feat_col])
            G_fam = build_graph(ids, sets, threshold, alpha=0.5)
            n_nf  = G_fam.number_of_nodes()
            n_ef  = G_fam.number_of_edges()
            densf = nx.density(G_fam) if n_nf > 1 else 0
            rows.append({
                'Característica': feat_name,
                'Umbral': threshold,
                'Alcance': 'Familia',
                'Familia': family,
                'Nodos': n_nf,
                'Aristas': n_ef,
                'Densidad': round(densf, 4),
            })

resumen = pd.DataFrame(rows)
print(resumen.to_string(index=False))
resumen

---
## 9. Análisis de Resultados

### Interpretación por umbral

| Umbral | Interpretación |
|--------|----------------|
| **0.3** | Similitud baja — grafos densos, muchas conexiones incluso entre muestras distintas. Útil para detectar cualquier relación. |
| **0.5** | Similitud moderada — balance entre conectividad y especificidad. |
| **0.7** | Similitud alta — solo las muestras más parecidas quedan conectadas. Muestra el núcleo de cada familia. |

### Diferencia entre características

- **API Calls**: compara funciones exactas. Dos muestras son similares solo si comparten muchas APIs idénticas. Más estricto.
- **Strings**: compara sub-tokens léxicos. Detecta similitudes entre muestras que usan funciones relacionadas de la misma familia (`Nt*`, `Reg*`, etc.) aunque no sean exactamente iguales. Más permisivo.

### Densidad del grafo

Una densidad alta dentro de una familia indica que las muestras de esa familia son **muy similares entre sí** → buena cohesión interna.  
Una densidad alta en el grafo global (mezclando familias) indica que las dos familias (Benigno / Malware) comparten características, lo que dificulta la separación.